# Casa 4. Tu propio documento

**Duración orientativa: 1 h**

Objetivo: repetir el pipeline completo con un PDF tuyo: parsing, deduplicación, chunking, preguntas,
versiones V3 y V7 y evaluación. Copia tu PDF en `data/mi_documento.pdf` (si no existe, se usa el del profesor).

In [1]:
from rag_lab import CFG, set_seeds, summary

my_pdf = CFG.DATA_DIR / "mi_documento.pdf"
CFG.DOC_PATH = my_pdf if my_pdf.exists() else CFG.DATA_DIR / "documento.pdf"
# Resultados, preguntas y etiquetas propios, para no mezclarlos con los de clase
CFG.QUESTIONS_PATH = CFG.DATA_DIR / f"preguntas_{CFG.DOC_PATH.stem}.jsonl"
CFG.RESULTS_CSV = CFG.DATA_DIR / f"results_{CFG.DOC_PATH.stem}.csv"
CFG.LABELS_PATH = CFG.DATA_DIR / f"etiquetas_{CFG.DOC_PATH.stem}.csv"
CFG.FAST_MODE = True
set_seeds()
summary()
print("Cachés de este documento en:", CFG.DOC_CACHE)

Documento:   /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/documento.pdf (existe)
FAST_MODE:   True
Dispositivo: cuda
GEN_MODEL:   google/gemini-3.1-flash-lite
JUDGE_MODEL: openai/gpt-5-mini
VLM_MODEL:   google/gemini-3.1-flash-lite
OpenRouter:  clave configurada


Trazas:      jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl  (alumno: profesor)
Cachés de este documento en: /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/cache/documento_4cad6871


/home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/rag_lab/tracing.py:49: UserWarning: No se pudo conectar con Langfuse ([Errno 111] Connection refused): las trazas se guardan en JSONL.
  warnings.warn(f"No se pudo conectar con Langfuse ({e}): las trazas se guardan en JSONL.")


In [2]:
import pandas as pd
from rag_lab import chunking, pdf

doc = pdf.parse()
clean_doc, removed = pdf.deduplicate(doc)
pages = pdf.clean_pages()
print(f"{pdf.num_pages(doc)} páginas, {len(doc.tables)} tablas, {len(doc.pictures)} imágenes; "
      f"{len(removed)} imágenes repetidas quitadas")
pd.DataFrame({n: chunking.summary(f(pages)) for n, f in chunking.STRATEGIES.items()}).T

20 páginas, 19 tablas, 28 imágenes; 26 imágenes repetidas quitadas


,fragmentos,media_chars,min_chars,max_chars,multipagina
fijo,113,764,143,800,20
recursivo,144,539,80,988,14
cabeceras,63,1256,95,5426,27


## Preguntas de evaluación

`questions.py` genera un borrador con 7 de texto, 7 de tabla y 6 visuales a partir de tu PDF. Si tu documento no
tiene tablas o elementos visuales, ajusta los números. Revisa el fichero antes de fiarte de las métricas.

In [3]:
from rag_lab import questions as question_gen

if CFG.QUESTIONS_PATH.exists():
    print("Ya existe", CFG.QUESTIONS_PATH.name, "(bórralo para regenerarlo)")
else:
    question_gen.generate_questions(n_text=7, n_table=7 if doc.tables else 0, n_visual=6)
from rag_lab import evaluation

questions = evaluation.load_questions()
pd.DataFrame(questions)[["id", "tipo", "paginas", "pregunta"]]

Ya existe preguntas_documento.jsonl (bórralo para regenerarlo)


,id,tipo,paginas,pregunta
0,q01,texto,[2],¿Qué línea se centra en la promoción del equil...
1,q02,texto,[3],Cuál es la duración total del proyecto según e...
2,q03,texto,[6],¿Cuál es el nombre del profesor del Centro de ...
3,q04,texto,[11],¿Cuántas fases principales componen el desarro...
4,q05,texto,[9],¿Qué centro de innovación y desarrollo para la...
5,q06,texto,[7],¿Cuál es la titulación del desarrollador backe...
6,q07,texto,[8],¿Cuál es el objetivo de la Tarea 9 dentro de l...
7,q08,tabla,[18],¿Qué instrumento se utiliza para medir el impa...
8,q09,tabla,[17],¿Qué instrumento se utiliza para medir el nive...
9,q10,tabla,[14],¿Cuántas empresas se beneficiarán por curso se...


In [4]:
from rag_lab import multimodal, results, versions

evaluation.evaluate_rag(versions.build("V3"), questions)
enriched = versions.enrich(versions.pdf_fragments("cabeceras"), versions.document_title(pages))
evaluation.evaluate_rag(multimodal.Multimodal(enriched, multimodal.describe_pages(), "siglip"), questions)
results.table("pdf")[["tecnica", "recall@5", "MRR@10", "recall@5_texto", "recall@5_tabla", "recall@5_visual",
                         "fidelidad", "latencia_p50_ms", "coste_usd"]].round(3)

,tecnica,recall@5,MRR@10,recall@5_texto,recall@5_tabla,recall@5_visual,fidelidad,latencia_p50_ms,coste_usd
version,,,,,,,,,
V3,Híbrido BM25 + denso con RRF,1.0,0.858,1.0,1.0,1.0,0.8,637.050,0.009
V7,V6 + unidades visuales (descripción VLM + SIGL...,1.0,0.935,1.0,1.0,1.0,0.9,3476.375,0.022


## Limitaciones de reutilizar el judge sin recalibrarlo

El judge se calibró (o se calibrará) con etiquetas humanas sobre el PDF de clase. Con tu documento:
- su acuerdo con una persona puede ser otro: el dominio, la jerga y el tipo de respuesta cambian;
- los fallos típicos cambian (por ejemplo, más cifras o más tablas) y el judge puede ser más permisivo con ellos;
- el prompt da por bueno "No aparece en el documento" como fiel, lo que premia respuestas evasivas si tu
  documento tiene muchas preguntas sin respuesta;
- las cifras de fidelidad no son comparables con las de clase hasta etiquetar una muestra propia y medir la kappa.

## Opcional: servicio ColPali

Si el profesor tiene levantado el servicio (ver `servidor/colpali/README.md`) y `COLPALI_URL` está en `.env`,
puedes consultarlo. El servicio indexa el PDF que tenga cargado el servidor (por defecto, el de clase).

In [5]:
if multimodal.colpali_available():
    for p in [q for q in questions if q["tipo"] == "visual"][:3]:
        print(p["pregunta"], "->", multimodal.colpali_search(p["pregunta"], 3))
else:
    print("Servicio ColPali no disponible: configura COLPALI_URL en .env para usarlo.")

Servicio ColPali no disponible: configura COLPALI_URL en .env para usarlo.


In [6]:
from rag_lab import models

models.release()